# NewVideoTranslate — 全本地 Colab GPU 视频翻译配音

**流程：**
1. 运行 Cell 1：挂载 Google Drive
2. 运行 Cell 2：开启 SSH 通道（运行完会打印 SSH 信息）
3. 运行 Cell 3：心跳保活（保持运行，不要关闭）
4. 把 SSH 信息 + YouTube 链接发给 Claude，剩下全自动

In [ ]:
# ============================================================
# Step 1: 挂载 Google Drive（用户手动运行）
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/video-translate'
os.makedirs(f'{DRIVE_ROOT}/processing', exist_ok=True)

# 从 .env 加载所有凭证
env_vars = {}
env_path = f'{DRIVE_ROOT}/.env'
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if '=' in line and not line.startswith('#'):
                key, val = line.split('=', 1)
                env_vars[key.strip()] = val.strip().strip("'\"")

HAS_COOKIES = os.path.exists(f'{DRIVE_ROOT}/youtube_cookies.txt')
HF_TOKEN = env_vars.get('HF_TOKEN')
NGROK_TOKEN = env_vars.get('NGROK_TOKEN')
SSH_PASSWORD = env_vars.get('SSH_PASSWORD')

# 验证必需凭证
missing = []
if not NGROK_TOKEN: missing.append('NGROK_TOKEN')
if not SSH_PASSWORD: missing.append('SSH_PASSWORD')
if missing:
    raise RuntimeError(f'Missing in .env: {", ".join(missing)}. See README for setup.')

print(f'Drive 挂载: ✓')
print(f'Cookies: {"✓" if HAS_COOKIES else "✗ (可选)"}')
print(f'HF_TOKEN: {"✓" if HF_TOKEN else "✗ (说话人分离需要)"}')
print(f'NGROK_TOKEN: ✓')
print(f'SSH_PASSWORD: ✓')

In [ ]:
# ============================================================
# Step 2: 开启 SSH 通道（用户手动运行）
# ============================================================
import subprocess, os, time

# 安装 SSH + ngrok
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'openssh-server'], capture_output=True)

# 设置 root 密码（从 .env 读取）
subprocess.run(f'echo "root:{SSH_PASSWORD}" | chpasswd', shell=True, capture_output=True)

# 配置 SSH
subprocess.run('sed -i "s/#PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config', shell=True, capture_output=True)
subprocess.run('sed -i "s/#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config', shell=True, capture_output=True)
subprocess.run(['service', 'ssh', 'start'], capture_output=True)

# 安装 pyngrok
subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
from pyngrok import ngrok

if not NGROK_TOKEN:
    raise RuntimeError('NGROK_TOKEN not found in .env! Please add it to Drive .env file.')
ngrok.set_auth_token(NGROK_TOKEN)

# 开启 TCP 隧道
tunnel = ngrok.connect(22, "tcp")
url = tunnel.public_url  # tcp://x.tcp.ngrok.io:PORT
host = url.replace("tcp://", "").split(":")[0]
port = url.split(":")[-1]

print(f'\n{"="*50}')
print(f'  SSH 通道已开启！')
print(f'  主机: {host}')
print(f'  端口: {port}')
print(f'  密码: {SSH_PASSWORD}')
print(f'{"="*50}')
print(f'\n把以下信息发给 Claude:')
print(f'  ssh root@{host} -p {port}')
print(f'  密码: {SSH_PASSWORD}')
print(f'\n✓ 此 cell 已完成，继续运行下一个 cell 启动心跳保活')

In [ ]:
# ============================================================
# Step 3: 心跳保活（用户手动运行，保持此 cell 运行）
# ============================================================
import time
from pyngrok import ngrok
from google.colab import output

# JS 心跳：模拟用户活动，防止前端空闲检测
output.eval_js('''
(function keepAlive() {
    setInterval(function() {
        google.colab.kernel.invokeFunction('_keepalive', [], {});
    }, 60000);
})();
''')

import google.colab
def _keepalive():
    return
google.colab.output.register_callback('_keepalive', _keepalive)

# Python 心跳循环：保持 cell 持续运行 + 监控隧道状态
print('⏳ 心跳保活已启动（每60秒），保持此页面打开...')
print('   按 Ctrl+C 或停止此 cell 可关闭\n')

try:
    tick = 0
    while True:
        time.sleep(60)
        tick += 1
        # 检查隧道是否还活着
        tunnels = ngrok.get_tunnels()
        if not tunnels:
            print(f'[{tick}m] ⚠️ 隧道断开，正在重连...')
            tunnel = ngrok.connect(22, "tcp")
            url = tunnel.public_url
            host = url.replace("tcp://", "").split(":")[0]
            port = url.split(":")[-1]
            print(f'[{tick}m] ✓ 新地址: ssh root@{host} -p {port}')
        else:
            if tick % 10 == 0:
                print(f'[{tick}m] ✓ 隧道正常，已运行 {tick} 分钟')
except KeyboardInterrupt:
    print('\n心跳已停止。')

---
### 以下由 Claude 通过 SSH 自动执行

保持此页面打开，不要关闭浏览器。Claude 会通过 SSH 完成：
- 安装依赖
- 人声分离 (Demucs)
- 转录 (WhisperX large-v3)
- 翻译 (Qwen2.5-7B-Instruct)
- TTS 配音 (Fish Speech S2 s2-pro)
- 视频合成 + P2 字幕烧录

完成后文件在 Google Drive: `MyDrive/video-translate/processing/{频道}/{标题}/video_chinese.mp4`